# Phase 1 Sanity Check
Verifies all preprocessed artifacts before PyG HeteroData construction.

**Checks:**
1. Index maps — all 10 node types: contiguity, no nulls, expected counts
2. Text embeddings — shape, dtype, L2 norms ≈ 1.0
3. Numeric blocks — shape, dtype, no NaN/inf, per-column ranges
4. Cross-artifact alignment — h5 rows == npy rows == idx map size, row-order consistency

In [1]:
import json
import numpy as np
import h5py
import pandas as pd
from pathlib import Path

DATA = Path("../data/processed")
pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_columns", 20)
print("Artifacts directory:", DATA.resolve())

Artifacts directory: C:\Users\wassim\Desktop\PFE Master\data\processed


---
## 1 · Index Maps

In [2]:
IDX_FILES = {
    "book":      "book_id_to_idx.json",
    "review":    "review_to_idx.json",
    "author":    "author_to_idx.json",
    "work":      "work_to_idx.json",
    "user":      "user_to_idx.json",
    "genre":     "genre_to_idx.json",
    "shelf":     "shelf_to_idx.json",
    "language":  "language_to_idx.json",
    "format":    "format_to_idx.json",
    "publisher": "publisher_to_idx.json",
}

idx_maps = {}
rows = []
issues = []

for node, fname in IDX_FILES.items():
    path = DATA / fname
    assert path.exists(), f"MISSING: {fname}"
    m = json.loads(path.read_text(encoding="utf-8"))
    idx_maps[node] = m

    n = len(m)
    indices = list(m.values())
    min_idx, max_idx = min(indices), max(indices)
    contiguous = (min_idx == 0 and max_idx == n - 1 and len(set(indices)) == n)
    empty_keys = sum(1 for k in m if k == "" or k is None)

    if not contiguous:
        issues.append(f"{node}: indices NOT contiguous!")
    if empty_keys:
        issues.append(f"{node}: {empty_keys} empty/null keys")

    rows.append({
        "node": node,
        "count": n,
        "min_idx": min_idx,
        "max_idx": max_idx,
        "contiguous": "✅" if contiguous else "❌",
        "empty_keys": empty_keys,
    })

df_idx = pd.DataFrame(rows).set_index("node")
display(df_idx)

if issues:
    print("\n⚠️  ISSUES:")
    for i in issues: print(" ", i)
else:
    print("\n✅ All index maps pass.")

,count,min_idx,max_idx,contiguous,empty_keys
node,,,,,
book,36514,0,36513,✅,0
review,154392,0,154391,✅,0
author,23105,0,23104,✅,0
work,25552,0,25551,✅,0
user,377799,0,377798,✅,0
genre,10,0,9,✅,0
shelf,60,0,59,✅,0
language,25,0,24,✅,0
format,12,0,11,✅,0



✅ All index maps pass.


---
## 2 · Text Embeddings

In [3]:
H5_FILES = {
    "book":   "books_bge_base_768.h5",
    "review": "reviews_bge_base_768.h5",
    "author": "authors_bge_base_768.h5",
    "work":   "works_bge_base_768.h5",
}

h5_data = {}  # node -> {ids, shape}
rows = []
issues = []

for node, fname in H5_FILES.items():
    path = DATA / fname
    assert path.exists(), f"MISSING: {fname}"
    with h5py.File(path, "r") as hf:
        emb = hf["embedding"][:]
        ids = [x.decode() for x in hf[list(hf.keys())[0]][:] if list(hf.keys())[0] != "embedding"]
        # find the id dataset
        id_key = [k for k in hf.keys() if k != "embedding"][0]
        ids = [x.decode() for x in hf[id_key][:]]

    norms = np.linalg.norm(emb, axis=1)
    nan_count = int(np.isnan(emb).sum())
    norm_ok = bool(np.allclose(norms, 1.0, atol=1e-4))

    h5_data[node] = {"ids": ids, "shape": emb.shape}

    if nan_count:
        issues.append(f"{node}: {nan_count} NaN values in embeddings")
    if not norm_ok:
        issues.append(f"{node}: L2 norms not ≈ 1.0 (min={norms.min():.4f}, max={norms.max():.4f})")

    rows.append({
        "node": node,
        "shape": str(emb.shape),
        "dtype": str(emb.dtype),
        "norm_mean": round(float(norms.mean()), 6),
        "norm_min":  round(float(norms.min()),  6),
        "norm_max":  round(float(norms.max()),  6),
        "L2≈1": "✅" if norm_ok else "❌",
        "NaN": nan_count,
    })

df_h5 = pd.DataFrame(rows).set_index("node")
display(df_h5)

if issues:
    print("\n⚠️  ISSUES:")
    for i in issues: print(" ", i)
else:
    print("\n✅ All embeddings pass.")

,shape,dtype,norm_mean,norm_min,norm_max,L2≈1,NaN
node,,,,,,,
book,"(36514, 768)",float32,1.0000,1.0000,1.0000,✅,0
review,"(154392, 768)",float32,1.0000,1.0000,1.0000,✅,0
author,"(23105, 768)",float32,1.0000,1.0000,1.0000,✅,0
work,"(25552, 768)",float32,1.0000,1.0000,1.0000,✅,0



✅ All embeddings pass.


---
## 3 · Numeric Blocks

In [4]:
NPY_FILES = {
    "book":   ("book_numeric.npy",   "book_numeric_columns.json"),
    "review": ("review_numeric.npy", "review_numeric_columns.json"),
    "author": ("author_numeric.npy", "author_numeric_columns.json"),
    "work":   ("work_numeric.npy",   "work_numeric_columns.json"),
}

npy_data = {}
issues = []

for node, (npy_fname, col_fname) in NPY_FILES.items():
    arr = np.load(DATA / npy_fname)
    cols = json.loads((DATA / col_fname).read_text())
    npy_data[node] = {"arr": arr, "cols": cols}

    nan_count = int(np.isnan(arr).sum())
    inf_count = int(np.isinf(arr).sum())
    if nan_count: issues.append(f"{node}: {nan_count} NaN in numeric")
    if inf_count: issues.append(f"{node}: {inf_count} Inf in numeric")
    if arr.shape[1] != len(cols): issues.append(f"{node}: col count mismatch ({arr.shape[1]} vs {len(cols)})")

    print(f"\n── {node.upper()} numeric  shape={arr.shape}  dtype={arr.dtype}  NaN={nan_count}  Inf={inf_count}")
    stats = pd.DataFrame(arr, columns=cols).describe().loc[["min","mean","max"]]
    display(stats)

if issues:
    print("\n⚠️  ISSUES:")
    for i in issues: print(" ", i)
else:
    print("\n✅ All numeric blocks pass.")


── BOOK numeric  shape=(36514, 9)  dtype=float32  NaN=0  Inf=0


,log_num_pages,year_normalized,avg_rating_filled,log_ratings_count,log_text_reviews_count,is_ebook_flag,is_num_pages_missing,is_year_missing,is_avg_rating_meaningful
min,0.0000,-18.2118,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
mean,4.8568,0.0086,4.0638,3.3543,1.7748,0.0782,0.2055,0.1593,0.9974
max,9.3021,163.9192,5.0000,13.8446,9.2499,1.0000,1.0000,1.0000,1.0000



── REVIEW numeric  shape=(154392, 4)  dtype=float32  NaN=0  Inf=0


,rating,log1p_n_votes,log1p_n_comments,is_rated
min,0.0000,0.0000,0.0000,0.0000
mean,3.8157,0.4435,0.0953,0.9565
max,5.0000,6.9717,5.1299,1.0000



── AUTHOR numeric  shape=(23105, 4)  dtype=float32  NaN=0  Inf=0


,average_rating,log1p_ratings_count,log1p_text_reviews_count,is_avg_rating_meaningful
min,0.0000,0.0000,0.0000,0.0000
mean,4.0374,5.0333,3.1798,0.9994
max,5.0000,16.1826,12.8144,1.0000



── WORK numeric  shape=(25552, 11)  dtype=float32  NaN=0  Inf=0


,original_year_normalized,log1p_books_count,log1p_ratings_count,log1p_text_reviews_count,rd_5_norm,rd_4_norm,rd_3_norm,rd_2_norm,rd_1_norm,is_year_missing,is_avg_rating_meaningful
min,-23.4462,0.6931,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
mean,0.0000,1.2530,3.7277,2.0195,0.4216,0.3173,0.1888,0.0536,0.0183,0.1024,0.9996
max,0.2558,7.5979,13.8512,9.3937,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000



✅ All numeric blocks pass.


---
## 4 · Cross-Artifact Alignment

In [5]:
# 4a. Row counts: h5 == npy == idx_map (where all three exist)
TRIPLETS = ["book", "author", "work"]  # review has h5+npy+idx but no npy before this session

rows = []
issues = []

for node in TRIPLETS:
    h5_n   = h5_data[node]["shape"][0]
    npy_n  = npy_data[node]["arr"].shape[0]
    idx_n  = len(idx_maps[node])
    ok = (h5_n == npy_n == idx_n)
    if not ok:
        issues.append(f"{node}: h5={h5_n}, npy={npy_n}, idx={idx_n} — MISMATCH")
    rows.append({"node": node, "h5_rows": h5_n, "npy_rows": npy_n, "idx_entries": idx_n, "match": "✅" if ok else "❌"})

# Review: h5 == npy == review_to_idx
rev_h5  = h5_data["review"]["shape"][0]
rev_npy = npy_data["review"]["arr"].shape[0]
rev_idx = len(idx_maps["review"])
ok = (rev_h5 == rev_npy == rev_idx)
if not ok: issues.append(f"review: h5={rev_h5}, npy={rev_npy}, idx={rev_idx} — MISMATCH")
rows.append({"node": "review", "h5_rows": rev_h5, "npy_rows": rev_npy, "idx_entries": rev_idx, "match": "✅" if ok else "❌"})

display(pd.DataFrame(rows).set_index("node"))

if issues:
    print("\n⚠️  ISSUES:"); [print(" ", i) for i in issues]
else:
    print("\n✅ Row counts consistent.")

,h5_rows,npy_rows,idx_entries,match
node,,,,
book,36514,36514,36514,✅
author,23105,23105,23105,✅
work,25552,25552,25552,✅
review,154392,154392,154392,✅



✅ Row counts consistent.


In [6]:
# 4b. Row-order: first and last ID in h5 must map to idx 0 and N-1 in the idx map
issues = []
rows = []

for node in ["book", "author", "work", "review"]:
    ids   = h5_data[node]["ids"]
    imap  = idx_maps[node]
    n     = len(ids)

    first_id, last_id = ids[0], ids[-1]
    first_ok = imap.get(first_id) == 0
    last_ok  = imap.get(last_id)  == n - 1

    # spot-check 3 random positions
    rng = np.random.default_rng(42)
    sample_idx = rng.integers(1, n-1, size=3).tolist()
    spot_ok = all(imap.get(ids[i]) == i for i in sample_idx)

    ok = first_ok and last_ok and spot_ok
    if not ok:
        details = []
        if not first_ok: details.append(f"first id maps to {imap.get(first_id)} not 0")
        if not last_ok:  details.append(f"last id maps to {imap.get(last_id)} not {n-1}")
        if not spot_ok:  details.append("spot checks failed")
        issues.append(f"{node}: " + "; ".join(details))

    rows.append({
        "node": node,
        "first_id": first_id[:20],
        "first→idx": imap.get(first_id),
        "last_id": last_id[:20],
        "last→idx": imap.get(last_id),
        "spot_checks": "✅" if spot_ok else "❌",
        "order_ok": "✅" if ok else "❌",
    })

display(pd.DataFrame(rows).set_index("node"))

if issues:
    print("\n⚠️  ISSUES:"); [print(" ", i) for i in issues]
else:
    print("\n✅ Row-order alignment confirmed.")

,first_id,first→idx,last_id,last→idx,spot_checks,order_ok
node,,,,,,
book,1000018,0,999914,36513,✅,✅
author,1000,0,999689,23104,✅,✅
work,1000008,0,999369,25551,✅,✅
review,28423ff309bc896c071a,0,726502e793d0f8df6461,154391,✅,✅



✅ Row-order alignment confirmed.


---
## 5 · Summary

In [7]:
rows = []

# U nodes — idx map only
for node in ["user", "genre", "shelf", "language", "format", "publisher"]:
    rows.append({"node": node, "category": "U", "idx_map": f"{len(idx_maps[node]):,}", "text_embed": "—", "numeric": "—"})

# TN nodes
for node in ["book", "review"]:
    h_sh = h5_data[node]["shape"]
    n_sh = npy_data[node]["arr"].shape
    rows.append({"node": node, "category": "TN",
                 "idx_map": f"{len(idx_maps[node]):,}",
                 "text_embed": f"{h_sh[0]:,} × {h_sh[1]}",
                 "numeric": f"{n_sh[0]:,} × {n_sh[1]}"})

# MN nodes
for node in ["author", "work"]:
    h_sh = h5_data[node]["shape"]
    n_sh = npy_data[node]["arr"].shape
    rows.append({"node": node, "category": "MN",
                 "idx_map": f"{len(idx_maps[node]):,}",
                 "text_embed": f"{h_sh[0]:,} × {h_sh[1]}",
                 "numeric": f"{n_sh[0]:,} × {n_sh[1]}"})

df_summary = pd.DataFrame(rows).set_index("node")
display(df_summary)
print("\n✅ Phase 1 preprocessing complete — ready for HeteroData construction.")

,category,idx_map,text_embed,numeric
node,,,,
user,U,"377,799",—,—
genre,U,10,—,—
shelf,U,60,—,—
language,U,25,—,—
format,U,12,—,—
publisher,U,86,—,—
book,TN,"36,514","36,514 × 768","36,514 × 9"
review,TN,"154,392","154,392 × 768","154,392 × 4"
author,MN,"23,105","23,105 × 768","23,105 × 4"



✅ Phase 1 preprocessing complete — ready for HeteroData construction.
